# Bài tập 3 — Đổi tham số làm trơn
Giữ cách chia dữ liệu như d3, thử alpha = 0.001, 0.1, 1, 10, 100 và giải thích.

Chọn **Run All** để chạy độc lập notebook này. Trên Colab, tải CSV cần dùng vào Files hoặc thư mục `data/`.


In [1]:
from pathlib import Path

# Chạy từ thư mục bài, code/, hoặc thư mục chứa CSV đã tải lên Colab.
thu_muc_hien_tai = Path.cwd().resolve()
thu_muc_du_an = thu_muc_hien_tai
for thu_muc in (thu_muc_hien_tai, *thu_muc_hien_tai.parents):
    if (thu_muc / "data" / "tin_nhan.csv").is_file():
        thu_muc_du_an = thu_muc
        break

def tep_du_lieu(ten):
    for duong_dan in (thu_muc_du_an / "data" / ten, thu_muc_hien_tai / ten):
        if duong_dan.is_file():
            return duong_dan
    raise FileNotFoundError(f"Không tìm thấy {ten}. Hãy giữ thư mục data/ hoặc tải CSV lên Colab.")

thu_muc_hinh = thu_muc_du_an / "figures"
thu_muc_hinh.mkdir(exist_ok=True)


## Mã và kết quả

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv(tep_du_lieu("tin_nhan.csv"))
# Giữ thứ tự vector hóa rồi chia như mã d3 của tài liệu để đối chiếu.
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

from sklearn.metrics import accuracy_score, precision_score, recall_score

ket_qua = []
print("Alpha       Accuracy   Precision   Recall")
for alpha in [0.001, 0.1, 1.0, 10.0, 100.0]:
    model = MultinomialNB(alpha=alpha).fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    pre = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    ket_qua.append((alpha, acc, pre, rec))
    print(f"{alpha:7.3f}     {acc:.4f}      {pre:.4f}     {rec:.4f}")
print("Nhận xét: Khi alpha quá lớn so với số đếm, các xác suất từ xích gần 1/V ở cả hai lớp, làm mất thông tin phân biệt của từ; quyết định dần phụ thuộc nhiều hơn vào tiên nghiệm.")
if len({tuple(row[1:]) for row in ket_qua}) == 1:
    print("Trong dãy alpha đã thử, cả ba số đo đều không đổi: các xác suất thay đổi nhưng chưa làm đổi kết quả phân loại trên tập kiểm tra này.")
else:
    print("Trong dãy alpha đã thử, các số đo có thay đổi; cần dựa vào bảng kết quả để so sánh, không kết luận alpha lớn hơn luôn tốt hơn.")

Alpha       Accuracy   Precision   Recall
  0.001     0.9467      1.0000     0.8667
  0.100     0.9467      1.0000     0.8667
  1.000     0.9467      1.0000     0.8667
 10.000     0.9467      1.0000     0.8667
100.000     0.9467      1.0000     0.8667
Nhận xét: Khi alpha quá lớn so với số đếm, các xác suất từ xích gần 1/V ở cả hai lớp, làm mất thông tin phân biệt của từ; quyết định dần phụ thuộc nhiều hơn vào tiên nghiệm.
Trong dãy alpha đã thử, cả ba số đo đều không đổi: các xác suất thay đổi nhưng chưa làm đổi kết quả phân loại trên tập kiểm tra này.
